In [19]:
import os
import json
import shutil
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from dtw import dtw
import pandas as pd

from collections import Counter



In [5]:
all_data = []

pose_indices = [0, 15, 16, 17, 18, 19, 20]
hand_indices = [0, 4, 7, 8, 11, 12, 15, 16, 19, 20]

In [7]:
def load_landmarks(filenames, num_frames, glossIndex = 4):
    
    
    # Prepare storage for data and labels
    video_data = []
    labels = []
    
    for file in filenames:
     
        # print(file)
        gloss = file.split('/')[glossIndex]
        # print(gloss)


        with open(file, 'rb') as f:
                landmarks = pickle.load(f)
                    
                # Prepare frame storage for each video with fixed number of frames
                frames = []

                # Get frame sampling step (skip frames if necessary)
                total_frames = len(landmarks)
                step = max(1, total_frames // num_frames)
                    
                # Process frames
                for i in range(num_frames):
                    frame_index = i * step if total_frames >= num_frames else i
                    
                    if frame_index < total_frames:
                        frame = landmarks[frame_index]
                    else:
                        frame = None
                        
                    # Extract and flatten required points
                    pose_points = frame['pose_landmarks'] if frame and frame['pose_landmarks'] else [None] * 33
                    left_hand_points = frame['left_hand_landmarks'] if frame and frame['left_hand_landmarks'] else [None] * 21
                    right_hand_points = frame['right_hand_landmarks'] if frame and frame['right_hand_landmarks'] else [None] * 21
                        
                    # Collect only specified indices and flatten them
                    extracted_points = []

                    # Collect pose points first
                    for idx in pose_indices:
                        point = pose_points[idx] if pose_points[idx] is not None else {'x': 0, 'y': 0}
                        extracted_points.extend([point['x'], point['y']])
                        
                    # print(left_hand_points)
                    for idx in hand_indices:
                        point = left_hand_points[idx] if left_hand_points[idx] is not None else {'x': 0, 'y': 0}
                        extracted_points.extend([point['x'], point['y']])

                    # Collect right hand points
                    for idx in hand_indices:
                        point = right_hand_points[idx] if right_hand_points[idx] is not None else {'x': 0, 'y': 0}
                        extracted_points.extend([point['x'], point['y']])
                        
                    frames.append(extracted_points)
                    
                # Add the processed video frames to the main array
                video_data.append(frames)
                labels.append(gloss)
                

    # Convert to numpy arrays
    video_data = np.array(video_data)
    labels = np.array(labels)
    
    return video_data, labels

In [ ]:


# Assume load_landmarks is already defined as in your existing code
# It takes file_names, num_frames, and optional glossIndex, returning video_data and labels

# Parameters
num_frames = 30  # Fixed number of frames per video, as in your example

# --- Step 1: Load the First Dataset ---
data_directory1 = '../all_outputs/output_ssl_small/'
file_names1 = []
for root, _, files in os.walk(data_directory1):
    for filename in files:
        file_names1.append(os.path.join(root, filename))
print(f"First dataset: {len(file_names1)} files found")
video_data1, labels1 = load_landmarks(file_names1, num_frames, glossIndex=3)

# --- Step 2: Load the Second Dataset ---
data_directory2 = '../all_outputs/output_include/'
file_names2 = []
for root, _, files in os.walk(data_directory2):
    for filename in files:
        file_names2.append(os.path.join(root, filename))
print(f"Second dataset: {len(file_names2)} files found")
video_data2, labels2 = load_landmarks(file_names2, num_frames, glossIndex=4)


First dataset: 960 files found
Second dataset: 4022 files found
Number of common glosses: 9
Gloss 'School': 20 videos in dataset1, 20 videos in dataset2
Gloss 'University': 20 videos in dataset1, 21 videos in dataset2
Gloss 'Tuesday': 20 videos in dataset1, 14 videos in dataset2
Gloss 'House': 20 videos in dataset1, 21 videos in dataset2
Gloss 'Today': 20 videos in dataset1, 14 videos in dataset2
Gloss 'Sister': 20 videos in dataset1, 20 videos in dataset2
Gloss 'Hello': 20 videos in dataset1, 21 videos in dataset2
Gloss 'Mother': 20 videos in dataset1, 20 videos in dataset2
Gloss 'Tomorrow': 20 videos in dataset1, 14 videos in dataset2


In [20]:
# --- Step 3: Select One Random Video per Gloss ---
def select_random_videos(video_data, labels, file_names):
    # Group indices by gloss
    gloss_to_indices = {}
    for idx, gloss in enumerate(labels):
        if gloss not in gloss_to_indices:
            gloss_to_indices[gloss] = []
        gloss_to_indices[gloss].append(idx)
    
    # Select one random index per gloss
    selected_indices = []
    selected_glosses = []
    selected_files = []
    for gloss, indices in gloss_to_indices.items():
        random_idx = random.choice(indices)  # Randomly pick one index
        selected_indices.append(random_idx)
        selected_glosses.append(gloss)
        selected_files.append(file_names[random_idx])
    
    # Extract corresponding video data
    selected_videos = video_data[selected_indices]
    return selected_videos, np.array(selected_glosses), selected_files

In [21]:
# Apply to both datasets
selected_videos1, selected_labels1, selected_files1 = select_random_videos(video_data1, labels1, file_names1)
selected_videos2, selected_labels2, selected_files2 = select_random_videos(video_data2, labels2, file_names2)
print(f"First dataset: {len(selected_labels1)} unique glosses selected")
print(f"Second dataset: {len(selected_labels2)} unique glosses selected")

# --- Step 4: Compute DTW for All Gloss Pairs ---
similarity_metrics = []

# Calculate total number of pairs for progress tracking
total_pairs = len(selected_videos1) * len(selected_videos2)
current_pair = 0

# Iterate over selected videos in dataset1
for idx1 in range(len(selected_videos1)):
    # Iterate over selected videos in dataset2
    for idx2 in range(len(selected_videos2)):
        # Increment pair counter
        current_pair += 1
        
        # Print current pair being processed with gloss names
        print(f"Processing pair {current_pair}/{total_pairs}: "
              f"gloss '{selected_labels1[idx1]}' vs gloss '{selected_labels2[idx2]}'")
        
        # Extract landmark sequences: shape (num_frames, num_features)
        seq1 = selected_videos1[idx1]
        seq2 = selected_videos2[idx2]
        
        # Compute DTW distance
        # Using dtw-python library, which handles multi-dimensional sequences with Euclidean distance by default
        alignment = dtw(seq1, seq2, keep_internals=True)
        distance = alignment.distance  # Lower distance means more similar movements
        
        # Store the result
        similarity_metrics.append({
            'gloss1': selected_labels1[idx1],
            'gloss2': selected_labels2[idx2],
            'video1': selected_files1[idx1],
            'video2': selected_files2[idx2],
            'dtw_distance': distance
        })

# --- Step 5: Sort and Save Results ---
# Convert to DataFrame
df = pd.DataFrame(similarity_metrics)

# Sort by dtw_distance in ascending order (most similar first)
df_sorted = df.sort_values(by='dtw_distance', ascending=True)

# Save to CSV
# df_sorted.to_csv('similarity_metrics_gloss_pairs.csv', index=False)
print(f"Saved {len(similarity_metrics)} similarity metrics to 'similarity_metrics_gloss_pairs.csv'")
print(f"Results sorted by DTW distance (lowest = most similar)")

First dataset: 48 unique glosses selected
Second dataset: 242 unique glosses selected
Processing pair 1/11616: gloss 'To_You' vs gloss 'Saturday'
Processing pair 2/11616: gloss 'To_You' vs gloss 'Yesterday'
Processing pair 3/11616: gloss 'To_You' vs gloss 'Tomorrow'
Processing pair 4/11616: gloss 'To_You' vs gloss 'Night'
Processing pair 5/11616: gloss 'To_You' vs gloss 'Second'
Processing pair 6/11616: gloss 'To_You' vs gloss 'Friday'
Processing pair 7/11616: gloss 'To_You' vs gloss 'Evening'
Processing pair 8/11616: gloss 'To_You' vs gloss 'Sunday'
Processing pair 9/11616: gloss 'To_You' vs gloss 'Week'
Processing pair 10/11616: gloss 'To_You' vs gloss 'Year'
Processing pair 11/11616: gloss 'To_You' vs gloss 'Hour'
Processing pair 12/11616: gloss 'To_You' vs gloss 'Tuesday'
Processing pair 13/11616: gloss 'To_You' vs gloss 'Minute'
Processing pair 14/11616: gloss 'To_You' vs gloss 'Wednesday'
Processing pair 15/11616: gloss 'To_You' vs gloss 'Today'
Processing pair 16/11616: gloss 'T

In [24]:
df_sorted[:30]

,gloss1,gloss2,video1,video2,dtw_distance
10395,Children,Price,../all_outputs/output_ssl_small/Children/1-2-w...,../all_outputs/output_include/Society/Price/MV...,56.667219
9675,Stay?,Ball,../all_outputs/output_ssl_small/Stay?/3-5-w054...,../all_outputs/output_include/Society/Ball/MVI...,59.543731
3141,Talk,Ball,../all_outputs/output_ssl_small/Talk/3-5-w0352...,../all_outputs/output_include/Society/Ball/MVI...,60.133106
9734,See?,Pencil,../all_outputs/output_ssl_small/See?/4-5-w0552...,../all_outputs/output_include/Home/Pencil/MVI_...,60.974418
3625,Report,Ball,../all_outputs/output_ssl_small/Report/4-1-w02...,../all_outputs/output_include/Society/Ball/MVI...,62.035024
1447,You,Ball,../all_outputs/output_ssl_small/You/4-4-w00220...,../all_outputs/output_include/Society/Ball/MVI...,63.375247
9917,See?,Ball,../all_outputs/output_ssl_small/See?/4-5-w0552...,../all_outputs/output_include/Society/Ball/MVI...,63.488235
9492,Stay?,Pencil,../all_outputs/output_ssl_small/Stay?/3-5-w054...,../all_outputs/output_include/Home/Pencil/MVI_...,63.616095
2958,Talk,Pencil,../all_outputs/output_ssl_small/Talk/3-5-w0352...,../all_outputs/output_include/Home/Pencil/MVI_...,63.791551
10224,Children,Letter,../all_outputs/output_ssl_small/Children/1-2-w...,../all_outputs/output_include/Home/Letter/MVI_...,63.832203


In [25]:
df_sorted.to_csv('similarity_metrics_gloss_pairs.csv', index=False)

In [ ]:
df_sorted.to_csv('similarity_metrics_gloss_pairs.csv', index=False)